In [2]:
import io
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2026/result/",
}

URL_TOKYO_2026 = "https://www.marathon.tokyo/2026/result/index.php"

# 1. Pede a Página 1 dos Homens (category=1)
payload_list = {
    "category": "1",
    "page": "1",
    "number": "",
    "name": "",
    "age": "",
    "country": "",
    "prefecture": "",
    "sort_key": "",
    "sort_asc": "",
    "d_number": "",
}

print(f"🔍 Consultando listagem de Tóquio 2026 (Masculino)...")
res = requests.post(URL_TOKYO_2026, headers=HEADERS, data=payload_list)
print(f"Status HTTP: {res.status_code}")

soup = BeautifulSoup(res.text, "html.parser")

# 2. Parsing das tabelas HTML
tables = pd.read_html(io.StringIO(res.text), flavor="lxml")
print(f"📊 Tabelas encontradas na página: {len(tables)}")

if tables:
    df_tokyo_page1 = tables[0]
    print("\n--- COLUNAS DA TABELA PRINCIPAL DE TÓQUIO ---")
    display(df_tokyo_page1.head(3))

# 3. Procura os links ou chamadas para os splits detalhados
print("\n--- IDENTIFICANDO LINKS / D_NUMBER DOS ATLETAS ---")
detail_triggers = []
for a in soup.find_all("a", href=True):
    # Checa se há links com d_number ou chamadas javascript
    if "d_number" in a["href"] or "javascript" in a["href"]:
        detail_triggers.append((a.text.strip(), a["href"]))

for txt, href in detail_triggers[:5]:
    print(f"• Atleta: '{txt}' -> Link/Trigger: '{href}'")

# 4. Se usa d_number no POST, testa o 1º atleta (BIB 5 - Tadese Takele)
payload_detail = {
    "category": "1",
    "page": "1",
    "d_number": "5",  # BIB do primeiro colocado
}
res_detail = requests.post(URL_TOKYO_2026, headers=HEADERS, data=payload_detail)
tables_detail = pd.read_html(io.StringIO(res_detail.text), flavor="lxml")

print(f"\n📊 Tabelas encontradas no perfil de splits: {len(tables_detail)}")
for i, tbl in enumerate(tables_detail):
    print(f"\n--- TABELA DE DETALHE [{i}] ---")
    display(tbl)

🔍 Consultando listagem de Tóquio 2026 (Masculino)...
Status HTTP: 200
📊 Tabelas encontradas na página: 2

--- COLUNAS DA TABELA PRINCIPAL DE TÓQUIO ---


,総合順位 Overall Place,参加種目 Race Category,ビブナンバー Bib Number,氏名 Name,年齢 Age,性別 Sex,国籍 Nationality,居住都市 City of Residence
0,1,マラソン男子 Marathon Men,5,ﾀﾃﾞｾ ﾀｹﾚ TADESE TAKELE,23,男性 Men,ETHIOPIA,-
1,2,マラソン男子 Marathon Men,6,ｼﾞｮﾌﾘｰ ﾄﾛｲﾃｨﾁ GEOFFREY TOROITICH,26,男性 Men,KENYA,-
2,3,マラソン男子 Marathon Men,2,ｱﾚｸｻﾝﾀﾞｰ ﾑﾃｨｿ ALEXANDER MUTISO,29,男性 Men,KENYA,-



--- IDENTIFICANDO LINKS / D_NUMBER DOS ATLETAS ---
• Atleta: '2' -> Link/Trigger: 'javascript:page(2);'
• Atleta: '3' -> Link/Trigger: 'javascript:page(3);'
• Atleta: '4' -> Link/Trigger: 'javascript:page(4);'
• Atleta: '5' -> Link/Trigger: 'javascript:page(5);'
• Atleta: '6' -> Link/Trigger: 'javascript:page(6);'

📊 Tabelas encontradas no perfil de splits: 2

--- TABELA DE DETALHE [0] ---


,総合順位 Overall Place,参加種目 Race Category,ビブナンバー Bib Number,氏名 Name,年齢 Age,性別 Sex,国籍 Nationality,居住都市 City of Residence
0,1,マラソン男子 Marathon Men,5,ﾀﾃﾞｾ ﾀｹﾚ TADESE TAKELE,23,男性 Men,ETHIOPIA,-
1,2,マラソン男子 Marathon Men,6,ｼﾞｮﾌﾘｰ ﾄﾛｲﾃｨﾁ GEOFFREY TOROITICH,26,男性 Men,KENYA,-
2,3,マラソン男子 Marathon Men,2,ｱﾚｸｻﾝﾀﾞｰ ﾑﾃｨｿ ALEXANDER MUTISO,29,男性 Men,KENYA,-
3,4,マラソン男子 Marathon Men,8,ﾀﾞﾆｴﾙ ﾏﾃｲｺ DANIEL MATEIKO,27,男性 Men,KENYA,-
4,5,マラソン男子 Marathon Men,15,ﾑｸﾀﾙ ｴﾄﾞﾘｽ MUKTAR EDRIS,32,男性 Men,ETHIOPIA,-
5,6,マラソン男子 Marathon Men,16,ｲﾘｱｽ ｱｵｱﾆ ILIASS AOUANI,30,男性 Men,ITALY,-
6,7,マラソン男子 Marathon Men,11,ｾﾚﾓﾝ ﾊﾞﾚｶﾞ SELEMON BAREGA,26,男性 Men,ETHIOPIA,-
7,8,マラソン男子 Marathon Men,9,ｾｲﾌ ﾄｩﾗ SEIFU TURA,28,男性 Men,ETHIOPIA,-
8,9,マラソン男子 Marathon Men,3,ｳﾞｨﾝｾﾝﾄ･ｷﾌﾟｹﾓｲ ｹﾞﾃｨｯﾁ VINCENT KIPKEMOI NGETICH,27,男性 Men,KENYA,-
9,10,マラソン男子 Marathon Men,12,ｼﾌｪﾗ ﾀﾑﾙ SHIFERA TAMRU,27,男性 Men,ETHIOPIA,-



--- TABELA DE DETALHE [1] ---


,0,1
0,参加種目／Race Category,選択してください／Select マラソン男子／Marathon Menマラソン女子／Mar...
1,ビブナンバー/Bib Number,NaN
2,氏名／Name,NaN
3,年代／Age,選択してください／Select 16～1920～2425～2930～3435～3940～4...
4,性別／Sex,男性／Men 女性／Women ノンバイナリー／Non-binary
5,国籍／Nationality,選択してください／Select 日本AFGHANISTANBAHRAINBANGLADES...
6,居住都市(都道府県)／City of Residence,選択してください 北海道／Hokkaido青森県／Aomori岩手県／Iwate宮城県／M...


In [3]:
# Inspeciona as células da primeira linha da tabela
primeira_linha = soup.select("table tr")[
    1
]  # Pula o cabeçalho (linha 0) e pega o 1º atleta
print("--- HTML DA LINHA DO 1º ATLETA ---")
print(primeira_linha.prettify())

# Procura todas as tags <a> e formulários dentro da tabela
print("\n--- LINKS E EVENTOS ENCONTRADOS NA LINHA ---")
for a in primeira_linha.find_all(["a", "button", "span", "input"]):
    print(
        f"Tag <{a.name}>: href='{a.get('href')}' | onclick='{a.get('onclick')}' | name='{a.get('name')}' | value='{a.get('value')}' | Texto='{a.text.strip()}'"
    )

--- HTML DA LINHA DO 1º ATLETA ---
<tr>
 <td class="taR">
  1
 </td>
 <td class="taC">
  マラソン男子
  <br/>
  Marathon Men
 </td>
 <td class="taR">
  5
 </td>
 <td>
  <a href="javascript:detail('5');">
   ﾀﾃﾞｾ ﾀｹﾚ
   <br/>
   TADESE TAKELE
  </a>
 </td>
 <td class="taC">
  23
 </td>
 <td class="taC">
  男性
  <br/>
  Men
 </td>
 <td class="taC">
  ETHIOPIA
 </td>
 <td class="taC">
  -
 </td>
</tr>


--- LINKS E EVENTOS ENCONTRADOS NA LINHA ---
Tag <a>: href='javascript:detail('5');' | onclick='None' | name='None' | value='None' | Texto='ﾀﾃﾞｾ ﾀｹﾚTADESE TAKELE'


In [4]:

# 1. Procura o código da função javascript:detail() nos scripts da página
print("--- CÓDIGO JAVASCRIPT DA FUNÇÃO DETAIL ---")
for script in soup.find_all("script"):
    if script.string and "detail" in script.string:
        for line in script.string.split("\n"):
            if any(
                k in line
                for k in ["function detail", "d_number", ".action", ".submit"]
            ):
                print(line.strip())

# 2. Inspeciona todos os formulários e seus campos de destino
print("\n--- FORMULÁRIOS (<form>) DA PÁGINA ---")
for i, f in enumerate(soup.find_all("form")):
    print(f"Form [{i}] -> Action: '{f.get('action')}' | Method: '{f.get('method')}' | Name: '{f.get('name')}'")
    for inp in f.find_all(["input", "select"]):
        print(f"   └─ {inp.name}: name='{inp.get('name')}' | value='{inp.get('value')}'")

--- CÓDIGO JAVASCRIPT DA FUNÇÃO DETAIL ---
form.submit();
form.submit();
form.submit();
function detail(number) {
form.action = "detail.php";
form.d_number.value = number;
form.submit();

--- FORMULÁRIOS (<form>) DA PÁGINA ---
Form [0] -> Action: 'index.php' | Method: 'post' | Name: 'form'
   └─ select: name='category' | value='None'
   └─ input: name='number' | value=''
   └─ input: name='name' | value=''
   └─ select: name='age' | value='None'
   └─ input: name='sex[]' | value='1'
   └─ input: name='sex[]' | value='2'
   └─ input: name='sex[]' | value='0'
   └─ select: name='country' | value='None'
   └─ select: name='prefecture' | value='None'
   └─ input: name='None' | value='検索／Search'
   └─ input: name='sort_key' | value='place'
   └─ input: name='sort_asc' | value='1'
   └─ input: name='page' | value='1'
   └─ input: name='d_number' | value=''


In [5]:
import io
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2026/result/index.php",
}

URL_DETAIL_2026 = "https://www.marathon.tokyo/2026/result/detail.php"

# Payload exato enviado pela função detail(number)
payload_detail = {
    "category": "1",  # 1 = Masculino, 2 = Feminino
    "d_number": "5",  # Número do atleta
    "page": "1",
    "sort_key": "place",
    "sort_asc": "1",
}

print(f"⏳ Requisitando parciais detalhadas em: {URL_DETAIL_2026}...")
res_detail = requests.post(URL_DETAIL_2026, headers=HEADERS, data=payload_detail)
print(f"Status HTTP: {res_detail.status_code}")

tables_tokyo = pd.read_html(io.StringIO(res_detail.text), flavor="lxml")
print(f"📊 Tabelas encontradas no perfil: {len(tables_tokyo)}\n")

for i, tbl in enumerate(tables_tokyo):
    print(f"--- TABELA DETALHE [{i}] ---")
    display(tbl)

⏳ Requisitando parciais detalhadas em: https://www.marathon.tokyo/2026/result/detail.php...
Status HTTP: 200
📊 Tabelas encontradas no perfil: 6

--- TABELA DETALHE [0] ---


,総合順位 Overall Place,ビブナンバー Bib Number,氏名 Name
0,1,5,ﾀﾃﾞｾ ﾀｹﾚ／TADESE TAKELE


--- TABELA DETALHE [1] ---


,0,1,2
0,参加種目／Race Category,マラソン男子／Marathon Men,参加種目別順位／Category Place： 1 ／27479
1,年齢／Age,23,年代別順位／Age Place (20～24)：1／634
2,性別／Sex,男性／Men,性別別順位／Gender Place：1／27479
3,国籍／Nationality,ETHIOPIA,国籍別順位／Nationality Place：1／16
4,タイム(ネット)／Time (net),02:03:36,02:03:36
5,タイム(グロス)／Time (gross),02:03:37,02:03:37


--- TABELA DETALHE [2] ---


...

,0,1,2
0,【参加種目／Category(マラソン／Marathon)】,1／37235,1／37235


--- TABELA DETALHE [4] ---


,0,1
0,参加種目／Race Category,選択してください／Select マラソン男子／Marathon Men


--- TABELA DETALHE [5] ---


,0,1
0,年代／Age,選択してください／Select 20～24
1,性別／Sex,男性／Men 女性／Women ノンバイナリー／Non-binary
2,国籍／Nationality,選択してください／Select ETHIOPIA
3,居住都市(都道府県)／City of Residence,選択してください／Select


In [6]:
print("🔍 Varrendo série histórica da Maratona de Tóquio (2007 a 2026)...\n")

anos_tokyo_disponiveis = []

for ano in range(2007, 2027):
    url_ano = f"https://www.marathon.tokyo/{ano}/result/index.php"
    payload_ano = {"category": "1", "page": "1"}

    try:
        r = requests.post(url_ano, headers=HEADERS, data=payload_ano, timeout=6)
        if r.status_code == 200:
            soup_ano = BeautifulSoup(r.text, "html.parser")
            cards = len(soup_ano.select("table tr"))
            if cards > 2:
                print(f"✅ Ano {ano}: DISPONÍVEL ({url_ano})")
                anos_tokyo_disponiveis.append(ano)
            else:
                print(f"⚪ Ano {ano}: Página existe mas sem tabela")
        else:
            print(f"⚪ Ano {ano}: HTTP {r.status_code}")
    except Exception as e:
        print(f"⚠️ Ano {ano}: Erro ({e})")

print(
    f"\n🏆 Total de anos disponíveis para Tóquio: {len(anos_tokyo_disponiveis)} edições ({anos_tokyo_disponiveis[0]} a {anos_tokyo_disponiveis[-1]})!"
)

🔍 Varrendo série histórica da Maratona de Tóquio (2007 a 2026)...

⚪ Ano 2007: HTTP 404
⚪ Ano 2008: HTTP 404
⚪ Ano 2009: HTTP 404
⚪ Ano 2010: HTTP 404
⚪ Ano 2011: HTTP 404
⚪ Ano 2012: HTTP 404
⚪ Ano 2013: HTTP 404
⚪ Ano 2014: Página existe mas sem tabela
✅ Ano 2015: DISPONÍVEL (https://www.marathon.tokyo/2015/result/index.php)
✅ Ano 2016: DISPONÍVEL (https://www.marathon.tokyo/2016/result/index.php)
✅ Ano 2017: DISPONÍVEL (https://www.marathon.tokyo/2017/result/index.php)
✅ Ano 2018: DISPONÍVEL (https://www.marathon.tokyo/2018/result/index.php)
✅ Ano 2019: DISPONÍVEL (https://www.marathon.tokyo/2019/result/index.php)
⚪ Ano 2020: HTTP 404
✅ Ano 2021: DISPONÍVEL (https://www.marathon.tokyo/2021/result/index.php)
⚪ Ano 2022: Página existe mas sem tabela
✅ Ano 2023: DISPONÍVEL (https://www.marathon.tokyo/2023/result/index.php)
✅ Ano 2024: DISPONÍVEL (https://www.marathon.tokyo/2024/result/index.php)
✅ Ano 2025: DISPONÍVEL (https://www.marathon.tokyo/2025/result/index.php)
✅ Ano 2026: DISPO

In [7]:
import io
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0"
}


def sondar_estatico_tokyo(ano: int):
    # Testa os arquivos de lista de homens (list1) e mulheres (list2)
    urls = [
        f"https://www.marathon.tokyo/{ano}/results/s_result_list1.html",
        f"https://www.marathon.tokyo/{ano}/result/s_result_list1.html",
        f"https://www.marathon.tokyo/{ano}/results/result_list1.html",
    ]

    for u in urls:
        try:
            r = requests.get(u, headers=HEADERS, timeout=6)
            if r.status_code == 200:
                tables = pd.read_html(io.StringIO(r.text), flavor="lxml")
                if tables:
                    df = tables[0]
                    return f"✅ Disponível ({len(df):,} linhas) -> URL: {u}"
        except Exception:
            continue
    return "⚪ Arquivo não localizado"


print("🔍 Inspecionando arquivos estáticos de Tóquio (2007 a 2014)...\n")
for ano_antigo in range(2007, 2015):
    status = sondar_estatico_tokyo(ano_antigo)
    print(f"• Ano {ano_antigo}: {status}")

🔍 Inspecionando arquivos estáticos de Tóquio (2007 a 2014)...

• Ano 2007: ✅ Disponível (1 linhas) -> URL: https://www.marathon.tokyo/2007/results/s_result_list1.html
• Ano 2008: ⚪ Arquivo não localizado
• Ano 2009: ⚪ Arquivo não localizado
• Ano 2010: ⚪ Arquivo não localizado
• Ano 2011: ⚪ Arquivo não localizado
• Ano 2012: ⚪ Arquivo não localizado
• Ano 2013: ⚪ Arquivo não localizado
• Ano 2014: ⚪ Arquivo não localizado


In [8]:
import io
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php?year=2008",
}

URL_HISTORICO_TOKYO = "https://www.marathon.tokyo/2014/index.php"

# 1. Teste de busca por BIB individual (BIB 2 em 2008)
payload_bib = {
    "holdingyear": "2008",
    "number": "2",
    "search_number": "ナンバーカード検索",
    "name": "",
}

print(
    "🔍 Testando busca de splits históricos via /2014/index.php (BIB 2 em 2008)..."
)
r_bib = requests.post(URL_HISTORICO_TOKYO, headers=HEADERS, data=payload_bib)
print(f"Status HTTP: {r_bib.status_code}")

tabelas_2008 = pd.read_html(io.StringIO(r_bib.text), flavor="lxml")
print(f"📊 Tabelas retornadas: {len(tabelas_2008)}")
for idx, tbl in enumerate(tabelas_2008):
    print(f"\n--- TABELA [{idx}] ---")
    display(tbl)

# 2. Teste de busca por Nome (para ver se devolve lista de corredores)
print("\n" + "=" * 60)
print("🔍 Testando busca por Nome (ex: name='a' ou name='tanaka')...")

payload_name = {
    "holdingyear": "2008",
    "number": "",
    "name": "a",  # Tenta buscar atletas com a letra 'a'
    "search_name": "姓名検索",
}
r_name = requests.post(URL_HISTORICO_TOKYO, headers=HEADERS, data=payload_name)
soup_name = BeautifulSoup(r_name.text, "html.parser")
tables_name = pd.read_html(io.StringIO(r_name.text), flavor="lxml")
print(
    f"📊 Tabelas retornadas na busca por nome: {len(tables_name)} | Links encontrados: {len(soup_name.find_all('a'))}"
)
if tables_name:
    display(tables_name[0].head(5))

🔍 Testando busca de splits históricos via /2014/index.php (BIB 2 em 2008)...
Status HTTP: 200
📊 Tabelas retornadas: 2

--- TABELA [0] ---


,0,1
0,氏名 NAME,ビクトル・ロスリン
1,ナンバー NUMBER,2
2,記録 TIME,2:07:23
3,チップタイム（参考） CHIP TIME,2:07:21
4,種目名 EVENT,マラソン男子 Men
5,順位 PLACE,1



--- TABELA [1] ---


,0,1,2,3
0,NaN,通過点 Point,通過タイム Intermediate time,スプリットタイム Split
1,NaN,5km,0:14:55,-
2,NaN,5km,0:14:55,0:15:04
3,NaN,10km,0:29:59,0:15:04
4,NaN,10km,0:29:59,0:15:04
5,NaN,15km,0:45:03,0:15:04
6,NaN,15km,0:45:03,0:15:16
7,NaN,20km,1:00:19,0:15:16
8,NaN,20km,1:00:19,0:15:16
9,NaN,中間点 Halfway Point,1:03:44,0:15:16



🔍 Testando busca por Nome (ex: name='a' ou name='tanaka')...
📊 Tabelas retornadas na busca por nome: 2 | Links encontrados: 12


,0,1
0,ナンバーカード(ゼッケン) Search by bib number,NaN


In [9]:
from bs4 import BeautifulSoup
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php?year=2008",
}

URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"

# 1. Teste enviando number em branco
payload_blank_num = {
    "holdingyear": "2008",
    "number": "",
    "search_number": "ナンバーカード検索",
}

# 2. Teste enviando name em branco
payload_blank_name = {
    "holdingyear": "2008",
    "name": "",
    "search_name": "姓名検索",
}

# 3. Teste enviando wildcard '%'
payload_wildcard = {
    "holdingyear": "2008",
    "name": "%",
    "search_name": "姓名検索",
}

for desc, payload in [
    ("Number em Branco", payload_blank_num),
    ("Name em Branco", payload_blank_name),
    ("Name Wildcard (%)", payload_wildcard),
]:
    r = requests.post(URL_HISTORICO, headers=HEADERS, data=payload, timeout=8)
    soup = BeautifulSoup(r.text, "html.parser")
    tables = soup.find_all("table")
    print(
        f"• {desc:<20} -> Status: HTTP {r.status_code} | Tabelas no HTML: {len(tables)}"
    )

• Number em Branco     -> Status: HTTP 200 | Tabelas no HTML: 2
• Name em Branco       -> Status: HTTP 200 | Tabelas no HTML: 2
• Name Wildcard (%)    -> Status: HTTP 200 | Tabelas no HTML: 2


In [10]:
import io
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php?year=2008",
}
URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"

payload_blank_name = {
    "holdingyear": "2008",
    "name": "",
    "search_name": "姓名検索",
}

r = requests.post(URL_HISTORICO, headers=HEADERS, data=payload_blank_name)
soup = BeautifulSoup(r.text, "html.parser")
tables = pd.read_html(io.StringIO(r.text), flavor="lxml")

print(f"📊 Total de Tabelas: {len(tables)}\n")
for idx, tbl in enumerate(tables):
    print(f"--- TABELA [{idx}] (Dimensões: {tbl.shape}) ---")
    display(tbl.head(10))

# Procura links ou paginação (ex: page=2, javascript:page(2), etc.)
print("\n--- LINKS E CONTROLES DE PÁGINA ENCONTRADOS ---")
for a in soup.find_all("a", href=True)[:10]:
    print(f"• Texto: '{a.text.strip()}' -> Href: '{a['href']}'")

📊 Total de Tabelas: 2

--- TABELA [0] (Dimensões: (1, 2)) ---


,0,1
0,ナンバーカード(ゼッケン) Search by bib number,NaN


--- TABELA [1] (Dimensões: (2, 2)) ---


,0,1
0,姓名,NaN
1,姓名,※ フルネームで入力してください。 ※ 外国人選手を検索する場合は姓名を半角アルファベット...



--- LINKS E CONTROLES DE PÁGINA ENCONTRADOS ---
• Texto: '' -> Href: 'javascript: st()'
• Texto: 'マラソン男子：Marathon/Men' -> Href: '/2008/results/s_result_list01.html'
• Texto: 'マラソン女子：Marathon/Women' -> Href: '/2008/results/s_result_list02.html'
• Texto: 'マラソン車いす男子／Marathon:Wheelchair(Men)' -> Href: '/2008/results/s_result_list03.html'
• Texto: 'マラソン車いす女子／Marathon:Wheelchair(Women)' -> Href: '/2008/results/s_result_list04.html'
• Texto: '10km一般男子／10km Race/Men' -> Href: '/2008/results/s_result_list05.html'
• Texto: '10km一般女子／10km Race/Women' -> Href: '/2008/results/s_result_list06.html'
• Texto: '10km車いす男子／10km Race:Wheelchair(Men)' -> Href: '/2008/results/s_result_list07.html'
• Texto: '10km視聴覚障害者男子･女子／10km Race:Visually Impaired Athletes(Men/Women)' -> Href: '/2008/results/s_result_list08.html'
• Texto: '10km知的障害者男子・女子／10km Race:Mentally Impaired Athletes(Men/Women)' -> Href: '/2008/results/s_result_list09.html'


In [11]:
import io
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php",
}
URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"


def testar_faixas_bib(ano: int):
    print(f"🔍 Testando faixas de BIB para Tóquio {ano}...")
    faixas = [
        ("Elite", [1, 2, 5, 10, 50, 100]),
        ("10k Range", [10001, 10100, 10500]),
        ("20k Range", [20001, 20100, 20500]),
        ("30k Range", [30001, 30100, 30500]),
        ("40k Range", [40001, 40100, 40500]),
        ("50k Range", [50001, 50100, 50500]),
        ("60k Range", [60001, 60100, 60500]),
        ("70k Range", [70001, 70100, 70500]),
    ]

    faixas_ativas = []
    for nome_faixa, bibs in faixas:
        encontrados = 0
        for b in bibs:
            payload = {
                "holdingyear": str(ano),
                "number": str(b),
                "search_number": "ナンバーカード検索",
                "name": "",
            }
            try:
                r = requests.post(
                    URL_HISTORICO, headers=HEADERS, data=payload, timeout=5
                )
                if "通過点" in r.text or "Point" in r.text:
                    encontrados += 1
            except Exception:
                pass
        if encontrados > 0:
            faixas_ativas.append(f"{nome_faixa} ({encontrados}/{len(bibs)})")

    print(f"   🎯 Faixas Ativas em {ano}: {', '.join(faixas_ativas)}\n")


for ano_teste in [2008, 2009, 2010, 2011, 2012, 2013, 2014]:
    testar_faixas_bib(ano_teste)

🔍 Testando faixas de BIB para Tóquio 2008...
   🎯 Faixas Ativas em 2008: 

🔍 Testando faixas de BIB para Tóquio 2009...
   🎯 Faixas Ativas em 2009: 

🔍 Testando faixas de BIB para Tóquio 2010...
   🎯 Faixas Ativas em 2010: 

🔍 Testando faixas de BIB para Tóquio 2011...
   🎯 Faixas Ativas em 2011: 

🔍 Testando faixas de BIB para Tóquio 2012...
   🎯 Faixas Ativas em 2012: 

🔍 Testando faixas de BIB para Tóquio 2013...
   🎯 Faixas Ativas em 2013: 

🔍 Testando faixas de BIB para Tóquio 2014...
   🎯 Faixas Ativas em 2014: 



In [12]:
import io
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php?year=2008",
    "Content-Type": "application/x-www-form-urlencoded",
}
URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"

# O valor exato que o PHP exige
SEARCH_BTN_RAW = "ナンバーカード検索"


def testar_faixas_com_payload_exato(ano: int):
    print(f"🔍 Testando faixas de BIB para Tóquio {ano}...")
    faixas = [
        ("Elite (1-500)", [1, 2, 5, 10, 50, 100, 183, 110]),
        ("10k Range", [10001, 10100, 10500]),
        ("20k Range", [20001, 20100, 20500]),
        ("30k Range", [30001, 30100, 30500]),
        ("40k Range", [40001, 40100, 40500]),
        ("50k Range", [50001, 50100, 50500]),
        ("60k Range", [60001, 60100, 60500]),
        ("70k Range", [70001, 70100, 70500]),
    ]

    faixas_ativas = []
    for nome_faixa, bibs in faixas:
        encontrados = 0
        for b in bibs:
            payload = {
                "holdingyear": str(ano),
                "number": str(b),
                "search_number": SEARCH_BTN_RAW,
                "name": "",
            }
            try:
                r = requests.post(
                    URL_HISTORICO, headers=HEADERS, data=payload, timeout=6
                )
                if (
                    r.status_code == 200
                    and ("通過点" in r.text or "Point" in r.text)
                    and "5km" in r.text
                ):
                    encontrados += 1
            except Exception:
                pass
        if encontrados > 0:
            faixas_ativas.append(f"{nome_faixa} ({encontrados}/{len(bibs)})")

    print(f"   🎯 Faixas Ativas em {ano}: {', '.join(faixas_ativas)}\n")


for ano_teste in [2008, 2009, 2010, 2011, 2012, 2013, 2014]:
    testar_faixas_com_payload_exato(ano_teste)

🔍 Testando faixas de BIB para Tóquio 2008...
   🎯 Faixas Ativas em 2008: 

🔍 Testando faixas de BIB para Tóquio 2009...
   🎯 Faixas Ativas em 2009: 

🔍 Testando faixas de BIB para Tóquio 2010...
   🎯 Faixas Ativas em 2010: 

🔍 Testando faixas de BIB para Tóquio 2011...
   🎯 Faixas Ativas em 2011: 

🔍 Testando faixas de BIB para Tóquio 2012...
   🎯 Faixas Ativas em 2012: 

🔍 Testando faixas de BIB para Tóquio 2013...
   🎯 Faixas Ativas em 2013: 

🔍 Testando faixas de BIB para Tóquio 2014...
   🎯 Faixas Ativas em 2014: 



In [13]:
import pandas as pd
import requests

URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"


def testar_com_referer_dinamico(ano: int):
    print(f"🔍 Testando faixas de BIB para Tóquio {ano}...")

    # Headers com o Referer amarrado dinamicamente ao ano
    headers_ano = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Content-Type": "application/x-www-form-urlencoded",
        "Origin": "https://www.marathon.tokyo",
        "Referer": f"https://www.marathon.tokyo/2014/index.php?year={ano}",
    }

    faixas = [
        ("Elite (1-500)", [1, 2, 5, 10, 50, 100, 183, 110]),
        ("1k-9k Range", [1001, 2001, 5001, 8001]),
        ("10k Range", [10001, 10100, 10500]),
        ("20k Range", [20001, 20100, 20500]),
        ("30k Range", [30001, 30100, 30500]),
        ("40k Range", [40001, 40100, 40500]),
        ("50k Range", [50001, 50100, 50500]),
        ("60k Range", [60001, 60100, 60500]),
        ("70k Range", [70001, 70100, 70500]),
    ]

    faixas_ativas = []
    for nome_faixa, bibs in faixas:
        encontrados = 0
        for b in bibs:
            # Envia a carga em formato de string codificada idêntica ao cURL
            payload_str = (
                f"holdingyear={ano}&number={b}&"
                f"search_number=%E3%83%8A%E3%83%B3%E3%83%90%E3%83%BC%E3%82%AB%E3%83%BC%E3%83%89%E6%A4%9C%E7%B4%A2&name="
            )
            try:
                r = requests.post(
                    URL_HISTORICO,
                    headers=headers_ano,
                    data=payload_str,
                    timeout=6,
                )
                if (
                    r.status_code == 200
                    and ("通過点" in r.text or "Point" in r.text)
                    and "5km" in r.text
                ):
                    encontrados += 1
            except Exception:
                pass
        if encontrados > 0:
            faixas_ativas.append(f"{nome_faixa} ({encontrados}/{len(bibs)})")

    print(f"   🎯 Faixas Ativas em {ano}: {', '.join(faixas_ativas)}\n")


for ano_teste in [2008, 2009, 2010, 2011, 2012, 2013, 2014]:
    testar_com_referer_dinamico(ano_teste)

🔍 Testando faixas de BIB para Tóquio 2008...
   🎯 Faixas Ativas em 2008: 

🔍 Testando faixas de BIB para Tóquio 2009...
   🎯 Faixas Ativas em 2009: 

🔍 Testando faixas de BIB para Tóquio 2010...
   🎯 Faixas Ativas em 2010: 

🔍 Testando faixas de BIB para Tóquio 2011...
   🎯 Faixas Ativas em 2011: 

🔍 Testando faixas de BIB para Tóquio 2012...
   🎯 Faixas Ativas em 2012: 

🔍 Testando faixas de BIB para Tóquio 2013...
   🎯 Faixas Ativas em 2013: 

🔍 Testando faixas de BIB para Tóquio 2014...
   🎯 Faixas Ativas em 2014: 



In [14]:
import io
import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Origin": "https://www.marathon.tokyo",
    "Referer": "https://www.marathon.tokyo/2014/index.php",
    "Content-Type": "application/x-www-form-urlencoded",
}
URL_HISTORICO = "https://www.marathon.tokyo/2014/index.php"

# 1. Inspeciona as opções do dropdown de anos
r_root = requests.get(URL_HISTORICO, headers=HEADERS, timeout=10)
r_root.encoding = r_root.apparent_encoding  # Corrige o encoding japonês
soup_root = BeautifulSoup(r_root.text, "html.parser")

select_holding = soup_root.find(
    "select", {"name": lambda n: n and "holding" in str(n).lower()}
)
anos_holding = []
if select_holding:
    for opt in select_holding.find_all("option"):
        anos_holding.append(opt.get("value", opt.text.strip()))

print(
    f"📅 ANOS ENCONTRADOS NO SELECT DE TÓQUIO HISTÓRICO:\n{anos_holding}\n"
)

# 2. Testa o atleta número 2 para cada ano com encoding corrigido
print("🔍 TESTANDO SPLITS DO ATLETA Nº 2 EM CADA ANO HISTÓRICO:\n")

for ano in ["2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014"]:
    payload = {
        "holdingyear": str(ano),
        "number": "2",
        "search_number": "ナンバーカード検索",
        "name": "",
    }
    headers_req = HEADERS.copy()
    headers_req["Referer"] = f"{URL_HISTORICO}?year={ano}"

    try:
        r = requests.post(
            URL_HISTORICO, headers=headers_req, data=payload, timeout=8
        )
        r.encoding = r.apparent_encoding  # Corrige o encoding

        tables = pd.read_html(io.StringIO(r.text), flavor="lxml")

        tem_splits = False
        nome_atleta = "-"
        tempo_finish = "-"

        for tbl in tables:
            # Tabela de metadados
            if len(tbl.columns) == 2 and any(
                "氏名" in str(v) or "NAME" in str(v) for v in tbl.values
            ):
                nome_atleta = str(tbl.iloc[0, 1]).strip()
                tempo_finish = (
                    str(tbl.iloc[2, 1]).strip() if len(tbl) > 2 else "-"
                )

            # Tabela de splits
            if any("5km" in str(v) for v in tbl.values) and any(
                "40km" in str(v) for v in tbl.values
            ):
                tem_splits = True

        status_splits = (
            "✅ 10 Splits Reais!"
            if tem_splits
            else "⚪ Sem tabela de splits"
        )
        print(
            f"• Ano {ano} (BIB 2) -> Status: {status_splits} | Atleta: '{nome_atleta}' ({tempo_finish})"
        )

    except Exception as e:
        print(f"• Ano {ano} (BIB 2) -> Erro: {e}")

📅 ANOS ENCONTRADOS NO SELECT DE TÓQUIO HISTÓRICO:
[]

🔍 TESTANDO SPLITS DO ATLETA Nº 2 EM CADA ANO HISTÓRICO:

• Ano 2007 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2008 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2009 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2010 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2011 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2012 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2013 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2014 (BIB 2) -> Erro: No tables found matching regex '.+'


In [16]:
import io
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Content-Type": "application/x-www-form-urlencoded",
    "Origin": "https://www.marathon.tokyo",
}

print("🔍 TESTANDO SPLITS HISTÓRICOS (2007 a 2014) COM URL E BODY SINCRONIZADOS:\n")

for ano in range(2007, 2015):
    url_ano = f"https://www.marathon.tokyo/2014/index.php?year={ano}"
    headers_ano = HEADERS.copy()
    headers_ano["Referer"] = url_ano

    # Payload exato em bytes com encoding do botão
    payload_raw = (
        f"holdingyear={ano}&number=2&"
        f"search_number=%E3%83%8A%E3%83%B3%E3%83%90%E3%83%BC%E3%82%AB%E3%83%BC%E3%83%89%E6%A4%9C%E7%B4%A2&name="
    )

    try:
        r = requests.post(
            url_ano, headers=headers_ano, data=payload_raw, timeout=10
        )
        r.encoding = r.apparent_encoding

        tables = pd.read_html(io.StringIO(r.text), flavor="lxml")

        nome = "-"
        tempo = "-"
        tem_splits = False

        for tbl in tables:
            # Metadados
            if len(tbl.columns) == 2 and any(
                "氏名" in str(v) or "NAME" in str(v) for v in tbl.values
            ):
                nome = str(tbl.iloc[0, 1]).strip()
                tempo = str(tbl.iloc[2, 1]).strip() if len(tbl) > 2 else "-"

            # Splits de 5k
            if any("5km" in str(v) for v in tbl.values) and any(
                "40km" in str(v) for v in tbl.values
            ):
                tem_splits = True

        status = "✅ 10 Splits Reais!" if tem_splits else "⚪ Sem splits"
        print(
            f"• Ano {ano} (BIB 2) -> Status: {status:<20} | Atleta: '{nome}' ({tempo})"
        )

    except Exception as e:
        print(f"• Ano {ano} (BIB 2) -> Erro: {e}")

🔍 TESTANDO SPLITS HISTÓRICOS (2007 a 2014) COM URL E BODY SINCRONIZADOS:

• Ano 2007 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2008 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2009 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2010 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2011 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2012 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2013 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2014 (BIB 2) -> Erro: No tables found matching regex '.+'


In [ ]:
import io
import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:153.0) Gecko/20100101 Firefox/153.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.marathon.tokyo",
}

POST_URL = "https://www.marathon.tokyo/2014/index.php"

print(
    "🔍 TESTANDO EXTRAÇÃO HISTÓRICA (2007 A 2014) COM SESSION HANDSHAKE:\n"
)

for ano in range(2007, 2015):
    sess = requests.Session()
    referer_url = f"https://www.marathon.tokyo/2014/index.php?year={ano}"

    headers_ano = HEADERS.copy()
    headers_ano["Referer"] = referer_url

    try:
        # 1. PASSO 1: Pré-voo GET para fixar a sessão e o ano
        r_pre = sess.get(referer_url, headers=headers_ano, timeout=10)

        # 2. PASSO 2: POST para a URL base (sem query string na URL)
        payload = {
            "holdingyear": str(ano),
            "number": "2",  # BIB 2
            "search_number": "ナンバーカード検索",
            "name": "",
        }

        r_post = sess.post(
            POST_URL, headers=headers_ano, data=payload, timeout=12
        )
        r_post.encoding = r_post.apparent_encoding

        tables = pd.read_html(io.StringIO(r_post.text), flavor="lxml")

        nome = "-"
        tempo = "-"
        tem_splits = False

        for tbl in tables:
            # Metadados
            if len(tbl.columns) == 2 and any(
                "氏名" in str(v) or "NAME" in str(v) for v in tbl.values
            ):
                nome = str(tbl.iloc[0, 1]).strip()
                tempo = str(tbl.iloc[2, 1]).strip() if len(tbl) > 2 else "-"

            # Splits de 5k
            if any("5km" in str(v) for v in tbl.values) and any(
                "40km" in str(v) for v in tbl.values
            ):
                tem_splits = True

        status = "✅ 10 Splits Reais!" if tem_splits else "⚪ Sem splits"
        print(
            f"• Ano {ano} (BIB 2) -> Status: {status:<20} | Atleta: '{nome}' ({tempo})"
        )

        sess.close()

    except Exception as e:
        print(f"• Ano {ano} (BIB 2) -> Erro: {e}")
        try:
            sess.close()
        except Exception:
            pass

🔍 TESTANDO EXTRAÇÃO HISTÓRICA (2007 A 2014) COM SESSION HANDSHAKE:

• Ano 2007 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2008 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2009 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2010 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2011 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2012 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2013 (BIB 2) -> Erro: No tables found matching regex '.+'
• Ano 2014 (BIB 2) -> Erro: No tables found matching regex '.+'
